<a href="https://colab.research.google.com/github/ApekshaBaviskar/Smart_ecommerce_project/blob/main/Smart_Ecommerce_ML_Colab_Cell_by_Cell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛒 Smart E-Commerce Intelligence System — Google Colab

This notebook is arranged so you can **run one cell at a time**.

### ML tasks
1. Predict **Annual Spending** — Regression
2. Predict **Customer Churn** — Classification
3. Compare Linear Regression, Decision Tree and Random Forest
4. Test a new customer

> The supplied CSV contains 200 rows and some missing values. This Colab version removes incomplete rows before modelling, leaving 145 usable rows.


## Cell 1 — Install / import libraries

In [ ]:
# Run this cell first
!pip -q install pandas numpy matplotlib seaborn scikit-learn
print("Libraries are ready!")


## Cell 2 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

sns.set_theme(style="whitegrid")
print("All libraries imported successfully!")


## Cell 3 — Upload the dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


## Cell 4 — Load the CSV

In [ ]:
# Select the customer CSV from the uploaded files
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("Please upload ecommerce_customers_200_clean.csv")

file_name = next(
    (name for name in csv_files if "customer" in name.lower()),
    csv_files[0]
)

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


## Cell 5 — Check the dataset

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

display(df.dtypes.to_frame("Data Type"))
display(df.isna().sum().to_frame("Missing Values"))


## Cell 6 — Clean the dataset

In [ ]:
# Remove rows containing missing values.
# This is necessary because the supplied CSV has missing values.

df = df.dropna().copy()

print("After cleaning:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))


## Cell 7 — Quick data exploration

In [ ]:
display(df.describe(include="all").T)


## Cell 8 — Visualize Annual Spending

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["Annual_Spending"], kde=True)
plt.title("Annual Spending Distribution")
plt.xlabel("Annual Spending (₹)")
plt.show()


## Cell 9 — Income vs Annual Spending

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="Income",
    y="Annual_Spending",
    hue="Membership_Type"
)
plt.title("Income vs Annual Spending")
plt.show()


## Cell 10 — Churn distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.xticks([0, 1], ["No Churn", "Churn"])
plt.title("Churn Distribution")
plt.show()


# PART A — REGRESSION
## Predict Annual Spending


## Cell 11 — Select regression features and target

In [ ]:
regression_features = [
    "Age", "Gender", "City_Tier", "Income", "Tenure_Months",
    "Visits_Per_Month", "Products_Bought", "Avg_Order_Value",
    "Discount_Usage_Rate", "Cart_Abandonment_Rate", "Reviews_Given",
    "Support_Tickets", "Days_Since_Last_Purchase",
    "Membership_Type", "Preferred_Category"
]

regression_target = "Annual_Spending"

X_reg = df[regression_features]
y_reg = df[regression_target]

categorical_features = [
    "Gender", "City_Tier", "Membership_Type", "Preferred_Category"
]

numeric_features = [
    "Age", "Income", "Tenure_Months", "Visits_Per_Month",
    "Products_Bought", "Avg_Order_Value", "Discount_Usage_Rate",
    "Cart_Abandonment_Rate", "Reviews_Given", "Support_Tickets",
    "Days_Since_Last_Purchase"
]

print("Number of features:", X_reg.shape[1])
print("Target:", regression_target)


## Cell 12 — Train/test split for regression

In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train_reg))
print("Testing rows:", len(X_test_reg))


## Cell 13 — Create preprocessing pipeline

In [ ]:
# Convert categorical columns into numbers.
# Numeric columns are passed through unchanged.

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

print("Preprocessing pipeline created.")


## Cell 14 — Linear Regression

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train_reg, y_train_reg)
linear_pred = linear_model.predict(X_test_reg)

linear_mae = mean_absolute_error(y_test_reg, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test_reg, linear_pred))
linear_r2 = r2_score(y_test_reg, linear_pred)

print("Linear Regression")
print(f"MAE  : ₹{linear_mae:,.2f}")
print(f"RMSE : ₹{linear_rmse:,.2f}")
print(f"R²   : {linear_r2:.3f}")


## Cell 15 — Decision Tree Regressor

In [ ]:
tree_reg_model = Pipeline([
    ("preprocessor", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
        remainder="passthrough"
    )),
    ("model", DecisionTreeRegressor(max_depth=5, random_state=42))
])

tree_reg_model.fit(X_train_reg, y_train_reg)
tree_pred = tree_reg_model.predict(X_test_reg)

tree_mae = mean_absolute_error(y_test_reg, tree_pred)
tree_rmse = np.sqrt(mean_squared_error(y_test_reg, tree_pred))
tree_r2 = r2_score(y_test_reg, tree_pred)

print("Decision Tree Regressor")
print(f"MAE  : ₹{tree_mae:,.2f}")
print(f"RMSE : ₹{tree_rmse:,.2f}")
print(f"R²   : {tree_r2:.3f}")


## Cell 16 — Random Forest Regressor

In [ ]:
forest_reg_model = Pipeline([
    ("preprocessor", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
        remainder="passthrough"
    )),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=42
    ))
])

forest_reg_model.fit(X_train_reg, y_train_reg)
forest_pred = forest_reg_model.predict(X_test_reg)

forest_mae = mean_absolute_error(y_test_reg, forest_pred)
forest_rmse = np.sqrt(mean_squared_error(y_test_reg, forest_pred))
forest_r2 = r2_score(y_test_reg, forest_pred)

print("Random Forest Regressor")
print(f"MAE  : ₹{forest_mae:,.2f}")
print(f"RMSE : ₹{forest_rmse:,.2f}")
print(f"R²   : {forest_r2:.3f}")


## Cell 17 — Compare regression models

In [ ]:
regression_results = pd.DataFrame([
    ["Linear Regression", linear_mae, linear_rmse, linear_r2],
    ["Decision Tree", tree_mae, tree_rmse, tree_r2],
    ["Random Forest", forest_mae, forest_rmse, forest_r2]
], columns=["Model", "MAE", "RMSE", "R²"])

display(regression_results.round(3))


## Cell 18 — Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test_reg, forest_pred, alpha=0.75)

min_value = min(y_test_reg.min(), forest_pred.min())
max_value = max(y_test_reg.max(), forest_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Annual Spending (₹)")
plt.ylabel("Predicted Annual Spending (₹)")
plt.title("Random Forest — Actual vs Predicted")
plt.show()


# PART B — CLASSIFICATION
## Predict Customer Churn


## Cell 19 — Select classification features and target

In [ ]:
classification_features = regression_features.copy()

X_clf = df[classification_features]
y_clf = df["Churn"]

print("Features:", X_clf.shape[1])
print("Target: Churn")
display(y_clf.value_counts().rename(index={0: "No Churn", 1: "Churn"}))


## Cell 20 — Train/test split for classification

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.20,
    random_state=42,
    stratify=y_clf
)

print("Training rows:", len(X_train_clf))
print("Testing rows:", len(X_test_clf))


## Cell 21 — Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
        remainder="passthrough"
    )),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

logistic_model.fit(X_train_clf, y_train_clf)

logistic_pred = logistic_model.predict(X_test_clf)
logistic_prob = logistic_model.predict_proba(X_test_clf)[:, 1]

print("Logistic Regression")
print(f"Accuracy : {accuracy_score(y_test_clf, logistic_pred):.3f}")
print(f"Precision: {precision_score(y_test_clf, logistic_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_test_clf, logistic_pred, zero_division=0):.3f}")
print(f"F1-score : {f1_score(y_test_clf, logistic_pred, zero_division=0):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_clf, logistic_prob):.3f}")


## Cell 22 — Decision Tree Classifier

In [ ]:
tree_clf_model = Pipeline([
    ("preprocessor", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
        remainder="passthrough"
    )),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
])

tree_clf_model.fit(X_train_clf, y_train_clf)

tree_clf_pred = tree_clf_model.predict(X_test_clf)
tree_clf_prob = tree_clf_model.predict_proba(X_test_clf)[:, 1]

print("Decision Tree Classifier")
print(f"Accuracy : {accuracy_score(y_test_clf, tree_clf_pred):.3f}")
print(f"Precision: {precision_score(y_test_clf, tree_clf_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_test_clf, tree_clf_pred, zero_division=0):.3f}")
print(f"F1-score : {f1_score(y_test_clf, tree_clf_pred, zero_division=0):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_clf, tree_clf_prob):.3f}")


## Cell 23 — Random Forest Classifier

In [ ]:
forest_clf_model = Pipeline([
    ("preprocessor", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
        remainder="passthrough"
    )),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42
    ))
])

forest_clf_model.fit(X_train_clf, y_train_clf)

forest_clf_pred = forest_clf_model.predict(X_test_clf)
forest_clf_prob = forest_clf_model.predict_proba(X_test_clf)[:, 1]

print("Random Forest Classifier")
print(f"Accuracy : {accuracy_score(y_test_clf, forest_clf_pred):.3f}")
print(f"Precision: {precision_score(y_test_clf, forest_clf_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_test_clf, forest_clf_pred, zero_division=0):.3f}")
print(f"F1-score : {f1_score(y_test_clf, forest_clf_pred, zero_division=0):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_clf, forest_clf_prob):.3f}")


## Cell 24 — Compare classification models

In [ ]:
classification_results = []

for name, pred, prob in [
    ("Logistic Regression", logistic_pred, logistic_prob),
    ("Decision Tree", tree_clf_pred, tree_clf_prob),
    ("Random Forest", forest_clf_pred, forest_clf_prob)
]:
    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_clf, pred),
        "Precision": precision_score(y_test_clf, pred, zero_division=0),
        "Recall": recall_score(y_test_clf, pred, zero_division=0),
        "F1": f1_score(y_test_clf, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test_clf, prob)
    })

classification_results = pd.DataFrame(classification_results)
display(classification_results.round(3))


## Cell 25 — Confusion matrix

In [ ]:
cm = confusion_matrix(y_test_clf, forest_clf_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cbar=False,
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

print("Confusion matrix:")
print(cm)


## Cell 26 — Random Forest feature importance

In [ ]:
rf_preprocessor = forest_clf_model.named_steps["preprocessor"]
rf_model = forest_clf_model.named_steps["model"]

feature_names = rf_preprocessor.get_feature_names_out()
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False).head(15)

display(importance_df)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top Random Forest Features for Churn Prediction")
plt.show()


## Cell 27 — Predict one customer

In [ ]:
sample_customer = df[regression_features].iloc[[0]].copy()

display(sample_customer)

predicted_spending = forest_reg_model.predict(sample_customer)[0]
predicted_churn_probability = forest_clf_model.predict_proba(sample_customer)[0, 1]

print(f"Predicted annual spending: ₹{predicted_spending:,.0f}")
print(f"Predicted churn probability: {predicted_churn_probability:.1%}")


## Cell 28 — Change a customer value and predict again

In [ ]:
student_customer = sample_customer.copy()

# Try changing this value:
student_customer["Days_Since_Last_Purchase"] = 60

display(student_customer)

new_spending = forest_reg_model.predict(student_customer)[0]
new_churn = forest_clf_model.predict_proba(student_customer)[0, 1]

print(f"New predicted spending: ₹{new_spending:,.0f}")
print(f"New churn probability: {new_churn:.1%}")


## Cell 29 — Final verification

In [ ]:
print("Final dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Unique Customer IDs:", int(df["Customer_ID"].nunique()))

print("\nML project completed successfully!")
